In [2]:
# Technical Pipeline: Transaction Fraud Detection Engine (v1.0)

import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import confusion_matrix, classification_report

np.random.seed(101)
n_tx = 1000
print("[INFO] Generating synthetic transaction database...")

tx_data = pd.DataFrame({
    'Tx_ID': range(1, n_tx + 1),
    'Amount_EUR': np.random.exponential(scale=100, size=n_tx),
    'Is_International': np.random.choice([0, 1], size=n_tx, p=[0.85, 0.15]),
    'Hour_Of_Day': np.random.randint(0, 24, n_tx)
})

# Bake in an over-aggressive rule: flags almost ALL late-night international transactions as fraud
def simulate_fraud(row):
    if row['Is_International'] == 1 and (row['Hour_Of_Day'] > 22 or row['Hour_Of_Day'] < 5):
        return 1 if np.random.rand() > 0.1 else 0
    return 1 if (row['Amount_EUR'] > 500 and np.random.rand() > 0.5) else 0

tx_data['Actual_Fraud'] = tx_data.apply(simulate_fraud, axis=1)

X = tx_data[['Amount_EUR', 'Is_International', 'Hour_Of_Day']]
y = tx_data['Actual_Fraud']
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=101)

fraud_model = RandomForestClassifier(n_estimators=10, random_state=101)
fraud_model.fit(X_train, y_train)

preds = fraud_model.predict(X_test)
tn, fp, fn, tp = confusion_matrix(y_test, preds).ravel()

print(f"\n" + "="*40)
print(f"--- FRAUD SYSTEM PERFORMANCE METRICS ---")
print(f"="*40)
print(f"Overall Classification Accuracy: {fraud_model.score(X_test, y_test)*100:.2f}%")
print(f"Legitimate Transactions Accidentally Frozen (False Positives): {fp}")
print(f"Actual Fraud Transactions Missed (False Negatives): {fn}")
print(f"Total Transactions Audited: {len(y_test)}")
print("="*40)

[INFO] Generating synthetic transaction database...

--- FRAUD SYSTEM PERFORMANCE METRICS ---
Overall Classification Accuracy: 99.00%
Legitimate Transactions Accidentally Frozen (False Positives): 1
Actual Fraud Transactions Missed (False Negatives): 1
Total Transactions Audited: 200
